In [1]:
# =====================================================================
# Cell 1 — Install packages (run once)
# =====================================================================
!pip install -q torch torchvision timm wandb huggingface_hub tqdm numpy pyyaml pandas matplotlib


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
# =====================================================================
# Cell 2 — Imports and global config
# =====================================================================
import os, json, time, zipfile, shutil, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import GradScaler, autocast          # new AMP API
from torch.utils.data import DataLoader
from torchvision import datasets

from timm.data import Mixup, create_transform
from timm.loss import SoftTargetCrossEntropy
from timm.scheduler import CosineLRScheduler
from timm.models.layers import DropPath, trunc_normal_

from torch.profiler import profile, ProfilerActivity
from huggingface_hub import hf_hub_download, login, whoami
from tqdm.auto import tqdm

# ------------------- CONFIG -------------------
DATA_ROOT       = "./data"
RAW_ROOT        = "./data/imagenet100_raw"
HF_REPO         = "asafaa/imagent100"
TRAIN_PARTS     = 5
VAL_PARTS       = 5
OUTPUT_DIR      = "./outputs/phase0"

EPOCHS          = 100
BATCH_SIZE      = 128
LR              = 1e-3          # ViG-Ti recipe
WEIGHT_DECAY    = 0.05
WARMUP_EPOCHS   = 10
GRAD_CLIP       = 1.0
NUM_WORKERS     = 8
NUM_CLASSES     = 100
SEED            = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

os.makedirs(DATA_ROOT,  exist_ok=True)
os.makedirs(RAW_ROOT,   exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Device:     {DEVICE}")
print(f"Data root:  {DATA_ROOT}")
print(f"Output dir: {OUTPUT_DIR}")
if torch.cuda.is_available():
    print(f"GPU:        {torch.cuda.get_device_name(0)}")

Device:     cuda
Data root:  ./data
Output dir: ./outputs/phase0
GPU:        NVIDIA H200


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/lib/python3.10/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [ ]:
# =====================================================================
# Cell 3 — HuggingFace login
# =====================================================================
HF_TOKEN = "hf_key"   # <-- replace with your own token
login(token=HF_TOKEN)
try:
    print(f"✓ Logged in as: {whoami()['name']}")
except Exception as e:
    print(f"⚠ whoami failed: {e}")

✓ Logged in as: Ranjan637


In [12]:
# =====================================================================
# Cell 4 — Download ImageNet-100 zip chunks + Labels.json
# =====================================================================
def download_all_chunks(repo_id, raw_root, train_parts=5, val_parts=5):
    os.makedirs(raw_root, exist_ok=True)
    downloaded = []
    for split, n_parts in [("train", train_parts), ("val", val_parts)]:
        for part in range(1, n_parts + 1):
            filename = f"imagenet100_{split}_part{part}.zip"
            print(f"Downloading {filename} ...")
            try:
                path = hf_hub_download(repo_id=repo_id, filename=filename,
                                       repo_type="dataset", local_dir=raw_root)
                downloaded.append(path)
                print(f"  ✓ saved to {path}")
            except Exception as e:
                print(f"  ✗ failed: {e}")
    try:
        labels_path = hf_hub_download(repo_id=repo_id, filename="Labels.json",
                                      repo_type="dataset", local_dir=raw_root)
        downloaded.append(labels_path)
        print(f"  ✓ Labels.json saved to {labels_path}")
    except Exception as e:
        print(f"  ⚠ Labels.json not found: {e}")
    return downloaded

downloaded_files = download_all_chunks(HF_REPO, RAW_ROOT, TRAIN_PARTS, VAL_PARTS)
print(f"\nTotal files downloaded: {len(downloaded_files)}")

  ✓ saved to /workspace/data/imagenet100_raw/imagenet100_train_part1.zip
  ✓ saved to /workspace/data/imagenet100_raw/imagenet100_train_part2.zip
  ✓ saved to /workspace/data/imagenet100_raw/imagenet100_train_part3.zip
  ✓ saved to /workspace/data/imagenet100_raw/imagenet100_train_part4.zip
  ✓ saved to /workspace/data/imagenet100_raw/imagenet100_train_part5.zip
  ✓ saved to /workspace/data/imagenet100_raw/imagenet100_val_part1.zip
  ✓ saved to /workspace/data/imagenet100_raw/imagenet100_val_part2.zip
  ✓ saved to /workspace/data/imagenet100_raw/imagenet100_val_part3.zip
  ✓ saved to /workspace/data/imagenet100_raw/imagenet100_val_part4.zip
  ✓ saved to /workspace/data/imagenet100_raw/imagenet100_val_part5.zip
  ✓ Labels.json saved to /workspace/data/imagenet100_raw/Labels.json

Total files downloaded: 11


In [13]:
# =====================================================================
# Cell 5 — Extract zips and merge into ImageFolder tree
# =====================================================================
def extract_and_merge(raw_root, data_root, train_parts=5, val_parts=5):
    tmp_dir = os.path.join(raw_root, "_extract_tmp")
    os.makedirs(tmp_dir, exist_ok=True)

    for split, n_parts in [("train", train_parts), ("val", val_parts)]:
        split_dir = os.path.join(data_root, split)
        os.makedirs(split_dir, exist_ok=True)

        for part in range(1, n_parts + 1):
            zip_name = f"imagenet100_{split}_part{part}.zip"
            zip_path = os.path.join(raw_root, zip_name)
            if not os.path.exists(zip_path):
                print(f"  ⚠ Missing {zip_name}, skipping")
                continue
            extract_target = os.path.join(tmp_dir, f"{split}_part{part}")
            os.makedirs(extract_target, exist_ok=True)
            print(f"Extracting {zip_name} -> {extract_target}")
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(extract_target)

            for root, dirs, files in os.walk(extract_target):
                if any(f.lower().endswith((".jpg", ".jpeg", ".png")) for f in files):
                    cls_name = os.path.basename(root)
                    dest = os.path.join(split_dir, cls_name)
                    os.makedirs(dest, exist_ok=True)
                    for f in files:
                        if f.lower().endswith((".jpg", ".jpeg", ".png")):
                            src = os.path.join(root, f)
                            dst = os.path.join(dest, f)
                            if not os.path.exists(dst):
                                shutil.move(src, dst)
    shutil.rmtree(tmp_dir, ignore_errors=True)
    print("\nExtraction + merge complete.")

extract_and_merge(RAW_ROOT, DATA_ROOT, TRAIN_PARTS, VAL_PARTS)

Extracting imagenet100_train_part1.zip -> ./data/imagenet100_raw/_extract_tmp/train_part1
Extracting imagenet100_train_part2.zip -> ./data/imagenet100_raw/_extract_tmp/train_part2
Extracting imagenet100_train_part3.zip -> ./data/imagenet100_raw/_extract_tmp/train_part3
Extracting imagenet100_train_part4.zip -> ./data/imagenet100_raw/_extract_tmp/train_part4
Extracting imagenet100_train_part5.zip -> ./data/imagenet100_raw/_extract_tmp/train_part5
Extracting imagenet100_val_part1.zip -> ./data/imagenet100_raw/_extract_tmp/val_part1
Extracting imagenet100_val_part2.zip -> ./data/imagenet100_raw/_extract_tmp/val_part2
Extracting imagenet100_val_part3.zip -> ./data/imagenet100_raw/_extract_tmp/val_part3
Extracting imagenet100_val_part4.zip -> ./data/imagenet100_raw/_extract_tmp/val_part4
Extracting imagenet100_val_part5.zip -> ./data/imagenet100_raw/_extract_tmp/val_part5

Extraction + merge complete.


In [4]:
# =====================================================================
# Cell 6 — Verify dataset
# =====================================================================
def verify_dataset(data_root):
    for split in ["train", "val"]:
        split_dir = os.path.join(data_root, split)
        if not os.path.isdir(split_dir):
            print(f"✗ {split_dir} does not exist"); continue
        classes = sorted([d for d in os.listdir(split_dir)
                          if os.path.isdir(os.path.join(split_dir, d))])
        n_images = sum(len(files) for _, _, files in os.walk(split_dir))
        print(f"{split}: {len(classes)} classes, {n_images} images")
        if len(classes) != NUM_CLASSES:
            print(f"  ⚠ Expected {NUM_CLASSES}, found {len(classes)}")

    train_classes = set(os.listdir(os.path.join(data_root, "train")))
    val_classes   = set(os.listdir(os.path.join(data_root, "val")))
    if train_classes != val_classes:
        print("⚠ Train/val class folders differ")
    else:
        print("\n✓ Train and val class folders match.")

    labels_path = os.path.join(RAW_ROOT, "Labels.json")
    if os.path.exists(labels_path):
        with open(labels_path) as f:
            labels = json.load(f)
        print(f"\nLabels.json loaded: {len(labels)} entries")
        for k, v in list(labels.items())[:5]:
            print(f"  {k} -> {v}")

verify_dataset(DATA_ROOT)

train: 100 classes, 128982 images
val: 100 classes, 5000 images

✓ Train and val class folders match.

Labels.json loaded: 100 entries
  n01498041 -> stingray
  n01629819 -> European fire salamander, Salamandra salamandra
  n01631663 -> eft
  n01641577 -> bullfrog, Rana catesbeiana
  n01644900 -> tailed frog, bell toad, ribbed toad, tailed toad, Ascaphus trui


In [5]:
# =====================================================================
# Cell 7 — CORRECTED Pyramid ViG-Ti model
# =====================================================================
# Fixes applied:
#   (1) LayerNorm instead of BatchNorm1d in MRConv2d
#   (2) Project BEFORE aggregate (matches official ViG)
#   (3) kNN graph built in FP32 (bypasses autocast)
#   (4) out = zeros(N, out_channels)  --- NOT zeros_like(x_j)
#   (5) edge_index is per-batch: row = b*N + i

class MRConv2d(nn.Module):
    """Max-Relative Graph Convolution (stable, project-then-aggregate)."""

    def __init__(self, in_channels, out_channels, act='relu', bias=True):
        super().__init__()
        self.act  = nn.ReLU() if act == 'relu' else nn.GELU()
        self.fc   = nn.Linear(in_channels, out_channels, bias=bias)
        self.norm = nn.LayerNorm(out_channels)

    def forward(self, x, edge_index):
        N = x.size(0)                        # <-- number of nodes, not edges
        row, col = edge_index
        x_j = x[row] - x[col]                # (E, C_in)
        x_j = self.act(x_j)
        x_j = self.fc(x_j)                   # (E, C_out)
        out = torch.zeros(N, x_j.size(-1),
                          device=x_j.device, dtype=x_j.dtype)   # <-- FIX #4
        out.index_add_(0, col, x_j)
        return self.norm(out)


class Grapher(nn.Module):
    def __init__(self, in_channels, kernel_size=9, dilation=1, drop_path=0.0):
        super().__init__()
        self.k = kernel_size
        self.dilation = dilation
        self.channels = in_channels
        self.graph_conv = MRConv2d(in_channels, in_channels * 2, act='relu')
        self.ffn = nn.Sequential(
            nn.Conv2d(in_channels * 2, in_channels * 4, 1, bias=False),
            nn.BatchNorm2d(in_channels * 4),
            nn.GELU(),
            nn.Conv2d(in_channels * 4, in_channels, 1, bias=False),
            nn.BatchNorm2d(in_channels),
        )
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()

    def _build_knn_graph(self, x):
        B, C, H, W = x.shape
        N = H * W
        k_eff = min(self.k, N - 1)
        if k_eff < 1:
            return torch.empty((2, 0), dtype=torch.long, device=x.device), B, N

        # FP32 similarity (bypass AMP entirely)
        with torch.amp.autocast('cuda', enabled=False):          # <-- FIX #3 (new API)
            x_flat = x.permute(0, 2, 3, 1).reshape(B, N, C).float()
            x_norm = F.normalize(x_flat, dim=-1, eps=1e-6)
            sim    = torch.bmm(x_norm, x_norm.transpose(1, 2))
            sim.diagonal(dim1=1, dim2=2).fill_(-float('inf'))
            _, knn_idx = sim.topk(k_eff, dim=-1)                 # (B, N, k_eff)

        # Per-batch offset so indices refer to x_flat rows [0, B*N)   <-- FIX #5
        base = (torch.arange(B, device=x.device).view(B, 1, 1) * N)      # (B,1,1)
        row  = torch.arange(N, device=x.device).view(1, N, 1) + base     # (B,N,1)
        row  = row.expand(B, N, k_eff)
        col  = knn_idx + base.expand(B, N, k_eff)                        # (B,N,k_eff)

        edge_index = torch.stack([row.reshape(-1), col.reshape(-1)], dim=0)
        return edge_index, B, N

    def forward(self, x):
        B, C, H, W = x.shape
        edge_index, B, N = self._build_knn_graph(x)
        x_flat = x.permute(0, 2, 3, 1).reshape(B * N, C)
        x_conv = self.graph_conv(x_flat, edge_index)
        x_conv = x_conv.reshape(B, H, W, -1).permute(0, 3, 1, 2)
        x_ffn  = self.ffn(x_conv)
        x      = self.drop_path(x_ffn) + x
        return x


class PyramidViG(nn.Module):
    def __init__(self, num_classes=1000, in_chans=3, embed_dim=48,
                 depths=(2, 2, 6, 2), drop_path_rate=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.depths = depths

        self.stem = nn.Sequential(
            nn.Conv2d(in_chans, embed_dim, kernel_size=16, stride=16, bias=False),
            nn.BatchNorm2d(embed_dim),
            nn.GELU(),
        )

        self.stages = nn.ModuleList()
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]
        cur = 0
        for i, depth in enumerate(depths):
            dim = embed_dim * (2 ** i)
            if i > 0:
                downsample = nn.Sequential(
                    nn.Conv2d(embed_dim * (2 ** (i - 1)), dim,
                              kernel_size=2, stride=2, bias=False),
                    nn.BatchNorm2d(dim),
                )
            else:
                downsample = nn.Identity()
            stage = nn.Sequential(
                downsample,
                *[Grapher(dim, kernel_size=9, drop_path=dpr[cur + j])
                  for j in range(depth)]
            )
            self.stages.append(stage)
            cur += depth

        self.norm = nn.BatchNorm2d(embed_dim * 8)
        self.head = nn.Linear(embed_dim * 8, num_classes)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward_features(self, x):
        x = self.stem(x)
        for stage in self.stages:
            x = stage(x)
        return x

    def forward(self, x):
        x = self.forward_features(x)
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        x = self.norm(x.unsqueeze(-1).unsqueeze(-1)).squeeze(-1).squeeze(-1)
        x = self.head(x)
        return x


def pyramid_vig_ti(num_classes=100):
    return PyramidViG(num_classes=num_classes, embed_dim=48,
                      depths=(2, 2, 6, 2), drop_path_rate=0.1)


# Quick sanity check
_m = pyramid_vig_ti(NUM_CLASSES)
print(f"Pyramid ViG-Ti params: {sum(p.numel() for p in _m.parameters() if p.requires_grad)/1e6:.2f}M")
del _m; torch.cuda.empty_cache()

Pyramid ViG-Ti params: 8.05M


In [6]:
# =====================================================================
# Cell 8 — Data loaders (DeiT-style augmentation)
# =====================================================================
def build_loaders(data_root, batch_size=128, num_workers=8):
    train_dir = os.path.join(data_root, "train")
    val_dir   = os.path.join(data_root, "val")

    train_transform = create_transform(
        input_size=224, is_training=True,
        color_jitter=0.4, auto_augment='rand-m9-mstd0.5-inc1',
        interpolation='bicubic', re_prob=0.25, re_mode='pixel', re_count=1,
    )
    val_transform = create_transform(
        input_size=224, is_training=False,
        interpolation='bicubic', crop_pct=0.95,
    )

    train_ds = datasets.ImageFolder(train_dir, transform=train_transform)
    val_ds   = datasets.ImageFolder(val_dir,   transform=val_transform)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True,
                              drop_last=True, persistent_workers=(num_workers > 0))
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True,
                              persistent_workers=(num_workers > 0))
    return train_loader, val_loader, len(train_ds.classes)

train_loader, val_loader, n_classes = build_loaders(DATA_ROOT, BATCH_SIZE, NUM_WORKERS)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Classes: {n_classes}")

Train batches: 1007 | Val batches: 40 | Classes: 100


In [7]:
# =====================================================================
# Cell 9 — Sanity check: forward pass produces no NaN and correct shape
# =====================================================================
def sanity_check(model, loader, device):
    model.train()
    xb, yb = next(iter(loader))
    xb, yb = xb.to(device), yb.to(device)
    print(f"Input batch: {tuple(xb.shape)}")
    with torch.no_grad(), autocast('cuda'):
        out = model(xb)
    out_f = out.float()
    print(f"Output shape: {tuple(out_f.shape)}")
    print(f"Output stats: mean={out_f.mean():.4f}  std={out_f.std():.4f}  "
          f"min={out_f.min():.4f}  max={out_f.max():.4f}  "
          f"nan={torch.isnan(out_f).any().item()}")

_m = pyramid_vig_ti(num_classes=NUM_CLASSES).to(DEVICE)
sanity_check(_m, train_loader, DEVICE)
del _m; torch.cuda.empty_cache()

Input batch: (128, 3, 224, 224)
Output shape: (128, 100)
Output stats: mean=-0.0000  std=0.3944  min=-1.6211  max=1.5420  nan=False


In [8]:
# =====================================================================
# Cell 10 — Experiment 0.1: Training + validation
# =====================================================================
def train_one_epoch(model, loader, optimizer, scaler, criterion, mixup_fn, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, targets in tqdm(loader, desc="Train", leave=False):
        images, targets = images.to(device, non_blocking=True), targets.to(device, non_blocking=True)
        if mixup_fn is not None:
            images, targets = mixup_fn(images, targets)
        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, targets)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * images.size(0)
        if mixup_fn is None:
            _, pred = outputs.max(1)
            correct += pred.eq(targets).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, targets in tqdm(loader, desc="Val", leave=False):
        images, targets = images.to(device, non_blocking=True), targets.to(device, non_blocking=True)
        with autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, targets)
        total_loss += loss.item() * images.size(0)
        _, pred = outputs.max(1)
        correct += pred.eq(targets).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total


def run_experiment_0_1():
    print("=" * 80)
    print("EXPERIMENT 0.1: Reproduce Pyramid ViG-Ti on ImageNet-100")
    print("=" * 80)

    model = pyramid_vig_ti(num_classes=NUM_CLASSES).to(DEVICE)
    print(f"Model params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M")

    mixup_fn = Mixup(mixup_alpha=0.8, cutmix_alpha=1.0, cutmix_minmax=None,
                     prob=1.0, switch_prob=0.5, mode='batch',
                     label_smoothing=0.1, num_classes=NUM_CLASSES)
    train_criterion = SoftTargetCrossEntropy()
    val_criterion   = nn.CrossEntropyLoss()

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler    = GradScaler('cuda')
    scheduler = CosineLRScheduler(optimizer, t_initial=EPOCHS, lr_min=1e-6,
                                  warmup_t=WARMUP_EPOCHS, warmup_lr_init=1e-6)

    best_acc, history = 0.0, []
    for epoch in range(EPOCHS):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scaler,
                                                train_criterion, mixup_fn, DEVICE)
        val_loss, val_acc = validate(model, val_loader, val_criterion, DEVICE)
        scheduler.step(epoch)
        elapsed = time.time() - t0

        history.append(dict(epoch=epoch+1, train_loss=train_loss, train_acc=train_acc,
                            val_loss=val_loss, val_acc=val_acc,
                            lr=optimizer.param_groups[0]['lr'], time=elapsed))
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss {train_loss:.4f} Acc {train_acc:.4f} | "
              f"Val Loss {val_loss:.4f} Acc {val_acc:.4f} | {elapsed:.1f}s")
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best_vig_ti.pth"))
            print(f"  -> New best: {best_acc:.4f}")

    print(f"\nBest validation accuracy: {best_acc:.4f}")
    pd.DataFrame(history).to_csv(os.path.join(OUTPUT_DIR, "training_history.csv"), index=False)
    return best_acc, history

In [9]:
# =====================================================================
# Cell 11 — Experiment 0.2: Profiling (no training)
# =====================================================================
def profile_model(model, input_size=(1, 3, 224, 224), device='cuda', warmup=10, runs=50):
    model.eval().to(device)
    dummy = torch.randn(*input_size, device=device)
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(dummy)
    torch.cuda.synchronize()

    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
                 record_shapes=True, profile_memory=True) as prof:
        with torch.no_grad():
            for _ in range(runs):
                _ = model(dummy)

    print("\n" + "=" * 80)
    print(f"PROFILING: Pyramid ViG-Ti | input={input_size}")
    print("=" * 80)
    print("\n--- Top 15 CUDA ops ---")
    print(prof.key_averages().table(sort_by='cuda_time_total', row_limit=15))

    # Component-level timings via monkey-patching
    timings = {'knn': [], 'graph_conv': [], 'ffn': [], 'total': []}
    original_forward = Grapher.forward

    def instrumented_forward(self, x):
        torch.cuda.synchronize()
        e0, e1, e2, e3, e4 = [torch.cuda.Event(enable_timing=True) for _ in range(5)]
        e0.record()
        B, C, H, W = x.shape
        edge_index, B, N = self._build_knn_graph(x)
        e1.record()
        x_flat = x.permute(0, 2, 3, 1).reshape(B * N, C)
        x_conv = self.graph_conv(x_flat, edge_index)
        x_conv = x_conv.reshape(B, H, W, -1).permute(0, 3, 1, 2)
        e2.record()
        x_ffn = self.ffn(x_conv)
        e3.record()
        x = self.drop_path(x_ffn) + x
        e4.record()
        torch.cuda.synchronize()
        timings['knn'].append(e0.elapsed_time(e1))
        timings['graph_conv'].append(e1.elapsed_time(e2))
        timings['ffn'].append(e2.elapsed_time(e3))
        timings['total'].append(e0.elapsed_time(e4))
        return x

    Grapher.forward = instrumented_forward
    with torch.no_grad():
        for _ in range(runs):
            _ = model(dummy)
    torch.cuda.synchronize()
    Grapher.forward = original_forward

    print("\n--- Average per-Grapher-layer timings (ms) ---")
    for k in ['knn', 'graph_conv', 'ffn', 'total']:
        arr = np.array(timings[k])
        print(f"  {k:12s}: {arr.mean():.3f} ms (std {arr.std():.3f})")

    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        _ = model(dummy)
    print(f"\nPeak GPU memory: {torch.cuda.max_memory_allocated()/1e6:.1f} MB")
    return prof, timings


def run_experiment_0_2():
    print("=" * 80)
    print("EXPERIMENT 0.2: Profile ViG-Ti (no training)")
    print("=" * 80)
    model = pyramid_vig_ti(num_classes=NUM_CLASSES)
    print(f"Model params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M")

    print("\n>>> Batch size 1")
    profile_model(model, input_size=(1, 3, 224, 224), device=DEVICE)
    print("\n>>> Batch size 32")
    profile_model(model, input_size=(32, 3, 224, 224), device=DEVICE)

In [10]:
# =====================================================================
# Cell 12 — RUN Phase 0
# =====================================================================
RUN_TRAINING  = True    # Experiment 0.1
RUN_PROFILING = False   # set True after training

if RUN_TRAINING:
    best_acc, history = run_experiment_0_1()

if RUN_PROFILING:
    prof, timings = run_experiment_0_2()

EXPERIMENT 0.1: Reproduce Pyramid ViG-Ti on ImageNet-100
Model params: 8.05M


Epoch 1/100 | Train Loss 4.6789 Acc 0.0000 | Val Loss 4.6684 Acc 0.0114 | 75.8s
  -> New best: 0.0114


Epoch 2/100 | Train Loss 4.6752 Acc 0.0000 | Val Loss 4.6751 Acc 0.0090 | 72.8s


Epoch 3/100 | Train Loss 4.6192 Acc 0.0000 | Val Loss 66.0502 Acc 0.0104 | 74.3s


Epoch 4/100 | Train Loss 4.5343 Acc 0.0000 | Val Loss 236.6111 Acc 0.0100 | 73.5s


Epoch 5/100 | Train Loss 4.4411 Acc 0.0000 | Val Loss 841.0340 Acc 0.0100 | 73.6s


... (19 outputs left out)

Epoch 25/100 | Train Loss 4.0299 Acc 0.0000 | Val Loss 4534.2780 Acc 0.0100 | 74.7s


Epoch 26/100 | Train Loss 4.0213 Acc 0.0000 | Val Loss 3162.4806 Acc 0.0100 | 73.8s


Epoch 27/100 | Train Loss 4.0222 Acc 0.0000 | Val Loss 4369.6832 Acc 0.0100 | 74.2s


KeyboardInterrupt: 

In [ ]:
# =====================================================================
# Cell 13 — Summary plots
# =====================================================================
print("=" * 80)
print("PHASE 0 — SUMMARY")
print("=" * 80)

if RUN_TRAINING and "best_acc" in globals():
    hist_path = os.path.join(OUTPUT_DIR, "training_history.csv")
    print(f"\n[0.1] Baseline reproduction:")
    print(f"  Best validation accuracy : {best_acc:.4f}")
    print(f"  Checkpoint               : {os.path.join(OUTPUT_DIR, 'best_vig_ti.pth')}")
    print(f"  History CSV              : {hist_path}")

    if os.path.exists(hist_path):
        df = pd.read_csv(hist_path)
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        axes[0].plot(df["epoch"], df["train_acc"], label="Train", marker="o", ms=3)
        axes[0].plot(df["epoch"], df["val_acc"],   label="Val",   marker="s", ms=3)
        axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy")
        axes[0].set_title("Top-1 Accuracy"); axes[0].grid(True, alpha=0.3); axes[0].legend()

        axes[1].plot(df["epoch"], df["train_loss"], label="Train", marker="o", ms=3)
        axes[1].plot(df["epoch"], df["val_loss"],   label="Val",   marker="s", ms=3)
        axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
        axes[1].set_title("Loss"); axes[1].grid(True, alpha=0.3); axes[1].legend()

        axes[2].plot(df["epoch"], df["lr"], color="tab:green")
        axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("LR")
        axes[2].set_title("LR Schedule"); axes[2].grid(True, alpha=0.3)

        plt.tight_layout()
        plot_path = os.path.join(OUTPUT_DIR, "phase0_training_curves.png")
        plt.savefig(plot_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"  Training curves saved at : {plot_path}")

if RUN_PROFILING and "timings" in globals():
    print(f"\n[0.2] Profiling:")
    for k in ["knn", "graph_conv", "ffn", "total"]:
        if k in timings and len(timings[k]) > 0:
            arr = np.array(timings[k])
            print(f"  {k:12s}: {arr.mean():.3f} ms (std {arr.std():.3f})")
    means = {k: np.mean(timings[k]) for k in ["knn", "graph_conv", "ffn"] if k in timings}
    if means and "knn" in means and "total" in timings:
        knn_pct = 100.0 * means["knn"] / np.mean(timings["total"])
        print(f"\n  >> Graph construction = {knn_pct:.1f}% of Grapher time")

print("\n" + "=" * 80)
print("Next: Phase 1 — Pilot Experiments")
print("=" * 80)